# Step G — Hydrogen gas network

**Task (Assignment 2, part g):** Assume that the countries of the Step D interconnected
system are also connected via gas pipelines transporting H₂ (or CH₄). Use a linear
approach to represent gas transport in pipelines. Optimise the network again and discuss
the results, including which of the two energy transport networks — electricity or H₂ —
is transporting more energy.

### Model

We extend the Step D interconnected system with a **parallel hydrogen network**:
- One H₂ bus per country (Denmark H2, Germany H2, Sweden H2, Norway H2)
- **Electrolysers** (extendable): electricity bus → H₂ bus, η = 70 % (alkaline)
- **Fuel cells** (extendable): H₂ bus → electricity bus, η = 58 %
- **H₂ pipelines** (fixed capacity): modelled as two directional `Link` per connection,
  each with efficiency 97 % (compression losses). Five connections are added, matching
  the electricity topology.

### Sources

- Electrolyser / fuel cell costs: DEA Technology Data, Renewable Fuels
- H₂ pipeline DK–DE capacity: European Hydrogen Backbone / Energinet (DHB1)
- Other pipeline capacities: assumed (no planned infrastructure)

**Requires in the working directory:** the same files as Step D
(`DK_2015_merged.csv`, `STEP D - electricity_demand.csv`, `Other country data/…`,
`functions_to_investigate.py`). The first code block rebuilds the Step D network
silently so that `net` is available for `net_g = net.copy()`.


## Imports

In [ ]:
from pathlib import Path
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import functions_to_investigate as fti


## Rebuild the Step D network (silent)

This block reproduces the Step D setup exactly, then optimises. After this, `net` is the
post-optimisation Step D network, ready to be copied and extended with an H₂ network.


In [ ]:
# ----- Cost table (same as Step A) -----
data = {
    "capital_cost": [
        1500000/25 + 60000,
        800000/25 + 14000,
        700000/25 + 24000,
    ],
    "marginal_cost": [0.0, 0.0, 9.5 * 3.6 / 0.56 + 2.30]
}
costs = pd.DataFrame(data, index=["wind_combined", "solar", "CCGT"])

# ----- Resolve paths -----
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent
cf_base = PROJECT_DIR / "Other country data" / "Energy charts data"

# ----- Denmark CFs + snapshot reference -----
dataframe_dk = pd.read_csv(PROJECT_DIR / "DK_2015_merged.csv", index_col=0, sep=",", parse_dates=True)
dataframe_dk.index = pd.to_datetime(dataframe_dk.index, utc=True).tz_localize(None)
dataframe_dk = dataframe_dk.sort_index()
CF_wind  = dataframe_dk["wind_cf_Unnamed: 1"].astype(float)
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"].astype(float)
snapshots = dataframe_dk.index

# ----- Multi-country demand -----
demand_all = pd.read_csv(PROJECT_DIR / "STEP D - electricity_demand.csv",
                         sep=";", index_col=0, parse_dates=True)
demand_all.index = pd.to_datetime(demand_all.index, utc=True).tz_localize(None)
demand_all = demand_all.sort_index()
demand_2015 = demand_all[demand_all.index.year == 2015].copy()
demand_dk = demand_2015["DNK"].astype(float).reindex(snapshots)
demand_de = demand_2015["DEU"].astype(float).reindex(snapshots)
demand_se = demand_2015["SWE"].astype(float).reindex(snapshots)
demand_no = demand_2015["NOR"].astype(float).reindex(snapshots)

# ----- Neighbour CFs -----
cf_de_raw = pd.read_csv(cf_base / "Germany" / "Germany_hourly_capacity_factors.csv")
cf_de_raw["timestamp"] = pd.to_datetime(cf_de_raw["timestamp"])
cf_de_raw = cf_de_raw.set_index("timestamp").sort_index()
cf_de_raw.index = cf_de_raw.index.tz_localize(None)
cf_de = cf_de_raw.resample("h").mean()
cf_de["wind_combined"] = cf_de["Wind onshore"]
cf_de["solar"] = cf_de["Solar AC"]
cf_de["CCGT"]  = cf_de["Fossil gas"]
cf_de["nuclear"] = cf_de["Nuclear"]
cf_de = cf_de[["wind_combined", "solar", "CCGT", "nuclear"]].reindex(snapshots)

cf_se = pd.read_csv(cf_base / "Sweden" / "Sweden_hourly_capacity_factors.csv")
cf_se["timestamp"] = pd.to_datetime(cf_se["timestamp"])
cf_se = cf_se.set_index("timestamp").sort_index()
cf_se.index = cf_se.index.tz_localize(None)
cf_se["wind_combined"] = cf_se["Wind onshore"]
cf_se["nuclear"] = cf_se["Nuclear"]
cf_se["hydro"]   = cf_se["Hydro water reservoir"]
cf_se = cf_se[["wind_combined", "nuclear", "hydro"]].reindex(snapshots).bfill()

cf_no = pd.read_csv(cf_base / "Norway" / "Norway_hourly_capacity_factors.csv")
cf_no["timestamp"] = pd.to_datetime(cf_no["timestamp"])
cf_no = cf_no.set_index("timestamp").sort_index()
cf_no.index = cf_no.index.tz_localize(None)
cf_no["wind_combined"] = cf_no["Wind onshore"]
cf_no["hydro"] = cf_no["Hydro water reservoir"]
cf_no = cf_no[["wind_combined", "hydro"]].reindex(snapshots)

# ----- Build the network -----
net = pypsa.Network()
net.set_snapshots(snapshots)
for name, (x, y) in {"Denmark": (10.0, 56.0), "Germany": (10.5, 51.5),
                     "Sweden": (15.0, 59.5), "Norway": (10.0, 62.0)}.items():
    net.add("Bus", name, x=x, y=y)

net.add("Load", "load_DK", bus="Denmark", p_set=demand_dk.values)
net.add("Load", "load_DE", bus="Germany", p_set=demand_de.values)
net.add("Load", "load_SE", bus="Sweden",  p_set=demand_se.values)
net.add("Load", "load_NO", bus="Norway",  p_set=demand_no.values)

net.add("Carrier", ["wind_onshore", "solar", "CCGT",
                    "hydro", "nuclear", "coal", "battery"],
        color=["blue", "yellow", "brown", "cyan", "purple", "grey", "purple"])

# Denmark — extendable
net.add("Generator", "DK_wind", bus="Denmark", carrier="wind_combined",
        capital_cost=costs.loc["wind_combined", "capital_cost"],
        marginal_cost=costs.loc["wind_combined", "marginal_cost"],
        p_max_pu=CF_wind.values, p_nom_extendable=True)
net.add("Generator", "DK_solar", bus="Denmark", carrier="solar",
        capital_cost=costs.loc["solar", "capital_cost"],
        marginal_cost=costs.loc["solar", "marginal_cost"],
        p_max_pu=CF_solar.values, p_nom_extendable=True)
net.add("Generator", "DK_CCGT", bus="Denmark", carrier="CCGT",
        capital_cost=costs.loc["CCGT", "capital_cost"],
        marginal_cost=costs.loc["CCGT", "marginal_cost"],
        efficiency=0.58, p_nom_extendable=True)

# Germany — fixed
net.add("Generator", "DE_wind", bus="Germany", carrier="wind_combined",
        p_nom=41300, marginal_cost=0, p_max_pu=cf_de["wind_combined"].values,
        p_nom_extendable=False)
net.add("Generator", "DE_solar", bus="Germany", carrier="solar",
        p_nom=37000, marginal_cost=0, p_max_pu=cf_de["solar"].values,
        p_nom_extendable=False)
net.add("Generator", "DE_CCGT", bus="Germany", carrier="CCGT",
        p_nom=28360, marginal_cost=60.0, p_nom_extendable=False)
net.add("Generator", "DE_nuclear", bus="Germany", carrier="nuclear",
        p_nom=10800, marginal_cost=10.0, p_nom_extendable=False)
net.add("Generator", "DE_coal", bus="Germany", carrier="coal",
        p_nom=21420, marginal_cost=30.0, p_nom_extendable=False)

# Sweden — fixed
net.add("Generator", "SE_hydro", bus="Sweden", carrier="hydro",
        p_nom=15920, marginal_cost=5.0, p_max_pu=cf_se["hydro"].values,
        p_nom_extendable=False)
net.add("Generator", "SE_nuclear", bus="Sweden", carrier="nuclear",
        p_nom=8900, marginal_cost=10.0, p_nom_extendable=False)
net.add("Generator", "SE_wind", bus="Sweden", carrier="wind_combined",
        p_nom=5500, marginal_cost=0, p_max_pu=cf_se["wind_combined"].values,
        p_nom_extendable=False)

# Norway — fixed
net.add("Generator", "NO_hydro", bus="Norway", carrier="hydro",
        p_nom=29900, marginal_cost=5.0, p_max_pu=cf_no["hydro"].values,
        p_nom_extendable=False)
net.add("Generator", "NO_wind", bus="Norway", carrier="wind_combined",
        p_nom=700, marginal_cost=0, p_max_pu=cf_no["wind_combined"].values,
        p_nom_extendable=False)

# Transmission lines
for bus in net.buses.index:
    net.buses.loc[bus, "v_nom"] = 380
for name, bus0, bus1, s_nom, length in [
    ("line_DK_DE", "Denmark", "Germany", 3500, 360),
    ("line_DK_SE", "Denmark", "Sweden",  1700, 520),
    ("line_DK_NO", "Denmark", "Norway",  1050, 570),
    ("line_SE_NO", "Sweden",  "Norway",  3500, 480),
    ("line_DE_SE", "Germany", "Sweden",   600, 820),
]:
    net.add("Line", name, bus0=bus0, bus1=bus1,
            s_nom=s_nom, x=0.1, r=0.01, length=length, p_nom=s_nom)

# Batteries (2024 Li-ion costs)
b_cap = 100_000 / 20 + 12_500 + (150_000 / 20) * 4   # 47,500 $/MW/year
for country, bus in [("DK", "Denmark"), ("DE", "Germany"),
                     ("SE", "Sweden"),  ("NO", "Norway")]:
    net.add("StorageUnit", f"{country}_battery", bus=bus, carrier="battery",
            capital_cost=b_cap, marginal_cost=0,
            efficiency_store=0.90**0.5, efficiency_dispatch=0.90**0.5,
            max_hours=4, cyclic_state_of_charge=True, p_nom_extendable=True)

# Optimise the Step D system
net.optimize(solver_name="highs", solver_options={"output_flag": False})
print(f"Step D rebuilt — system cost: {net.objective/1e9:.3f} B$/y")


## H₂ technology assumptions

In [ ]:
# H2 technology assumptions

# Electrolyser (alkaline, 2025 estimates)
electrolyser_investment = 600000   # $/MW
electrolyser_lifetime   = 25       # years
electrolyser_fom        = 12000    # $/MW/year (2% of investment)
electrolyser_capex      = electrolyser_investment / electrolyser_lifetime + electrolyser_fom  # 36,000 $/MW/y
electrolyser_efficiency = 0.70     # electricity → H2 (LHV)

# Fuel cell / H2 turbine (reconversion H2 → electricity)
fuelcell_investment = 700000       # $/MW
fuelcell_lifetime   = 25           # years
fuelcell_fom        = 24000        # $/MW/year
fuelcell_capex      = fuelcell_investment / fuelcell_lifetime + fuelcell_fom  # 52,000 $/MW/y
fuelcell_efficiency = 0.58         # H2 → electricity

# H2 pipeline efficiency (compression losses)
pipeline_efficiency = 0.97

# H2 pipeline capacities [MW]
h2_pipeline_cap = {
    "H2_DK_DE": 3600,   # EHB / Energinet (DHB1)
    "H2_DK_SE": 1750,   # assumed
    "H2_DK_NO": 1100,   # assumed
    "H2_SE_NO": 3600,   # assumed
    "H2_DE_SE": 650,   # assumed
}

print("--- Step G: H2 technology assumptions ---")
print(f"Electrolyser:  {electrolyser_capex/1e3:.0f} k$/MW/y, η = {electrolyser_efficiency:.0%}")
print(f"Fuel cell:     {fuelcell_capex/1e3:.0f} k$/MW/y, η = {fuelcell_efficiency:.0%}")
print(f"Pipeline η:    {pipeline_efficiency:.0%}")


## Copy the Step D network and add the H₂ layer

In [ ]:
# Copy the Step D network and add H2 carrier + buses

net.model.solver_model = None
net_g = net.copy()

net_g.add("Carrier", "H2", color="#2ecc71")

countries = ["Denmark", "Germany", "Sweden", "Norway"]
for c in countries:
    net_g.add("Bus", f"{c} H2", carrier="H2")

print(f"Buses: {list(net_g.buses.index)}")


In [ ]:
# Add electrolysers: electricity bus → H2 bus (extendable)

for c in countries:
    net_g.add("Link", f"{c} electrolyser",
              bus0=c,
              bus1=f"{c} H2",
              carrier="H2",
              capital_cost=electrolyser_capex,
              efficiency=electrolyser_efficiency,
              p_nom_extendable=True)

print("Electrolysers added:", [f"{c} electrolyser" for c in countries])


In [ ]:
# Add fuel cells: H2 bus → electricity bus (extendable)

for c in countries:
    net_g.add("Link", f"{c} fuel cell",
              bus0=f"{c} H2",
              bus1=c,
              carrier="H2",
              capital_cost=fuelcell_capex,
              efficiency=fuelcell_efficiency,
              p_nom_extendable=True)

print("Fuel cells added:", [f"{c} fuel cell" for c in countries])


In [ ]:
# Add H2 pipelines (fixed capacity, two links per connection for each direction)

h2_connections = [
    ("DK", "DE", "Denmark H2", "Germany H2", 3600),   # EHB / Energinet
    ("DK", "SE", "Denmark H2", "Sweden H2",  1500),   # assumed
    ("DK", "NO", "Denmark H2", "Norway H2",  2000),   # assumed
    ("SE", "NO", "Sweden H2",  "Norway H2",  2000),   # assumed
    ("DE", "SE", "Germany H2", "Sweden H2",  1000),   # assumed
]

for c0, c1, bus0, bus1, cap in h2_connections:
    # Forward: bus0 → bus1
    net_g.add("Link", f"H2_{c0}_{c1}",
              bus0=bus0, bus1=bus1,
              carrier="H2",
              p_nom=cap,
              efficiency=pipeline_efficiency,
              p_nom_extendable=False)
    # Backward: bus1 → bus0
    net_g.add("Link", f"H2_{c1}_{c0}",
              bus0=bus1, bus1=bus0,
              carrier="H2",
              p_nom=cap,
              efficiency=pipeline_efficiency,
              p_nom_extendable=False)

print("H2 pipelines added (2 links per connection):")
for c0, c1, _, _, cap in h2_connections:
    print(f"  {c0} ↔ {c1}: {cap} MW (each direction)")


## Optimise the combined electricity + H₂ network

In [ ]:
# Optimize the combined electricity + H2 network

net_g.optimize(solver_name="highs", 
               solver_options={"output_flag": False},
               progress=False)

print(f"System cost Step D: {net.objective/1e9:.3f} B$/y")
print(f"System cost Step G: {net_g.objective/1e9:.3f} B$/y")
print(f"Difference:         {(net_g.objective - net.objective)/1e9:+.3f} B$/y")


# Results: electrolyser and fuel cell capacities

print("--- Electrolyser capacities [MW] ---")
for c in countries:
    cap = net_g.links.loc[f"{c} electrolyser", "p_nom_opt"]
    print(f"  {c}: {cap:.1f} MW")

print("\n--- Fuel cell capacities [MW] ---")
for c in countries:
    cap = net_g.links.loc[f"{c} fuel cell", "p_nom_opt"]
    print(f"  {c}: {cap:.1f} MW")


## Which network transports more energy?

In [ ]:
# Results: compare electricity vs H2 total annual energy transport

# Electricity: sum of |flow| on all HVAC lines
el_transport_twh = net_g.lines_t.p0.abs().sum().sum() / 1e6

# H2 pipelines: filter links that start with "H2_"
h2_pipe_names = [n for n in net_g.links.index if n.startswith("H2_")]
h2_transport_twh = net_g.links_t.p0[h2_pipe_names].abs().sum().sum() / 1e6

print("--- Energy transport comparison ---")
print(f"Electricity network: {el_transport_twh:.2f} TWh/y")
print(f"H2 pipeline network: {h2_transport_twh:.2f} TWh/y")
print(f"Ratio H2/Electricity: {h2_transport_twh/el_transport_twh:.2%}")

if h2_transport_twh > el_transport_twh:
    print("→ The H2 network transports MORE energy than the electricity network.")
else:
    print("→ The electricity network transports MORE energy than the H2 network.")


## Impact on Denmark: capacity comparison Step D vs Step G

In [ ]:
# Results: Denmark capacity comparison Step D vs Step G

print("--- Denmark optimal capacities: Step D vs Step G ---")
dk_gens_d = net.generators[net.generators.bus == "Denmark"]
dk_gens_g = net_g.generators[net_g.generators.bus == "Denmark"]

for gen_d, gen_g in zip(dk_gens_d.index, dk_gens_g.index):
    cap_d = dk_gens_d.loc[gen_d, "p_nom_opt"] / 1e3
    cap_g = dk_gens_g.loc[gen_g, "p_nom_opt"] / 1e3
    carrier = dk_gens_d.loc[gen_d, "carrier"]
    print(f"  {carrier:15s}: {cap_d:.2f} → {cap_g:.2f} GW")


## Plots

In [ ]:
importlib.reload(fti)
fti.plot_energy_transport_comparison(net_g)


In [ ]:
fti.plot_h2_infrastructure(net_g)


In [ ]:
fti.plot_h2_pipeline_utilization(net_g)
